# Alice: symbolic and numerical optimization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gromicho/teaching/blob/main/foundations/optimization/alice-optimization.ipynb) [![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/gromicho/teaching/main?urlpath=tree/foundations/optimization/alice-optimization.ipynb)

**Worked Example** · UvA Business Analytics teaching collection

Work through the questions before running each cell. Explain the result, check its assumptions, and change one input to test your understanding.


## Setup
Install missing symbolic and plotting libraries first. Pyomo and Ipopt are introduced after the symbolic calculation and plot.


In [ ]:
# Load the shared teaching utilities from this checkout or a verified download.
from pathlib import Path
import hashlib
import sys
from urllib.request import urlopen

support_path = next((folder / 'support' for folder in [Path.cwd(), *Path.cwd().parents]
                     if (folder / 'support' / 'teaching_utils.py').is_file()), None)
if support_path is None:
    support_path = Path.cwd() / '.teaching-support'
    support_path.mkdir(exist_ok=True)
    helper = support_path / 'teaching_utils.py'
    expected = 'fbfa41e12709a01548e213abba976dad1e21d0abcfd798a2dfd669706cc152bb'
    if not helper.exists() or hashlib.sha256(helper.read_bytes()).hexdigest() != expected:
        url = 'https://raw.githubusercontent.com/gromicho/teaching/f3ad11b77cae7dd05315d314c7e72ee8516aaa3d/support/teaching_utils.py'
        content = urlopen(url, timeout=45).read()
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError('Teaching helper version changed; reopen the current course notebook.')
        helper.write_bytes(content)
sys.path.insert(0, str(support_path))
from teaching_utils import ensure_packages

required_packages = {'matplotlib': 'matplotlib', 'numpy': 'numpy', 'scipy': 'scipy', 'sympy': 'sympy'}
ensure_packages(required_packages)


## A stable vase
Alice's teaching model describes the centre of gravity as a function of the water height $h$:
$$c(h)=\frac{4\pi h^2+2000}{8\pi h+200},\qquad 0\leq h\leq20.$$
Minimizing this function gives the most stable height within this simplified model. The original photograph is not redistributed because its image licence was not established. The mathematical example does not require it.

Before computing, sketch the trade-off: why could neither an empty nor a full vase be best?


In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
h = sp.Symbol('h', real=True)
c = (4*sp.pi*h**2+2000)/(8*sp.pi*h+200)
derivative = sp.diff(c,h)
stationary = sp.solve(derivative,h)
stationary


In [ ]:
candidates = [0.0,20.0] + [float(v) for v in stationary if v.is_real and 0 <= float(v) <= 20]
objective = sp.lambdify(h,c,'numpy')
best = min(candidates,key=objective)
print('Candidates:', candidates, 'best height:', best)
result = minimize_scalar(objective,bounds=(0,20),method='bounded')
assert result.success
assert abs(result.x-best) < 1e-4


In [ ]:
grid = np.linspace(0,20,201)
plt.plot(grid,objective(grid))
plt.scatter([best],[objective(best)],color='red')
plt.xlabel('Water height h')
plt.ylabel('Centre-of-gravity height c(h)')
plt.show()


## Express and solve the same objective in Pyomo
Now introduce Pyomo and the nonlinear solver Ipopt. Pyomo expresses the model; Ipopt solves it. Start from a neutral value and compare the computed solution with the symbolic result and SciPy. HiGHS does not solve this rational nonlinear objective.


In [ ]:
from teaching_utils import ensure_packages, install_coin_solvers, solve_checked
ensure_packages({'pyomo': 'pyomo'})
install_coin_solvers()
import pyomo.environ as pyo
alice = pyo.ConcreteModel('Alice')
alice.h = pyo.Var(bounds=(0, 20), initialize=10)
alice.cog = pyo.Objective(expr=(4*np.pi*alice.h**2+2000)/(8*np.pi*alice.h+200))
ipopt_result = solve_checked(alice, 'ipopt')
assert abs(pyo.value(alice.h) - best) < 1e-5
assert abs(pyo.value(alice.h) - result.x) < 1e-4
assert abs(pyo.value(alice.cog) - float(objective(best))) < 1e-8
alice.display()


## Why is this a minimum?
Compare every stationary point inside the bounds with the two endpoints. Explain what a derivative test establishes, and what the bounds add. A successful numerical termination alone is not a proof of global optimality for an arbitrary nonlinear problem.
